# Capa Silver: limpieza de procesos de contratación

En esta etapa tomamos los datos crudos de la capa Bronze y los dejamos listos para analizar.
El objetivo es tener **una fila por cada proceso de contratación**, con datos confiables y en el formato correcto.

**Resultado:** tres tablas
- `procesos`: los procesos limpios, listos para análisis.
- `procesos_cuarentena`: los registros que no pasaron las reglas de calidad, guardados aparte para revisarlos.
- `adjudicaciones`: qué proveedores ganó cada proceso.

## 1. Eliminar registros repetidos

La fuente entrega algunas filas exactamente iguales, repetidas varias veces. Las eliminamos porque no aportan información y distorsionarían cualquier conteo.

También identificamos los procesos que tienen **más de un valor de adjudicación**. En esos casos la fuente mezcla varios proveedores y valores, y no es posible saber qué valor corresponde a cuál proveedor.

In [0]:
# Configuración y deduplicación
from pyspark.sql import functions as F

CATALOG = "workspace_practica_1"
b = spark.table(f"{CATALOG}.secop_bronze.procesos").dropDuplicates()

valores = (b.groupBy("id_del_proceso")
             .agg(F.countDistinct("valor_total_adjudicacion").alias("n_valores")))

## 2. Organizar y validar los procesos

Aquí dejamos cada proceso en una sola fila y corregimos el formato de los datos: las fechas pasan a ser fechas, los valores en pesos pasan a ser números, y los nombres de las entidades quedan escritos de forma uniforme.

Cuando un proceso tiene un valor de adjudicación ambiguo, **lo dejamos vacío y lo marcamos**, en lugar de elegir uno al azar. Es preferible no tener el dato que tener un dato equivocado.

**Reglas de calidad:** un proceso es válido si tiene fecha de publicación y un precio base mayor que cero. Los que no cumplen van a la tabla de cuarentena, no se borran.

In [0]:
# Tabla de procesos, tipada
procesos = (b.join(valores, "id_del_proceso")
  .select(
    "id_del_proceso",
    F.upper(F.trim("entidad")).alias("entidad"),
    "nit_entidad", "ordenentidad", "ciudad_entidad",
    F.col("modalidad_de_contratacion").alias("modalidad"),
    "tipo_de_contrato", "estado_del_procedimiento", "adjudicado",
    F.to_date("fecha_de_publicacion_del").alias("fecha_publicacion"),
    F.col("precio_base").cast("decimal(20,2)").alias("precio_base"),
    F.when(F.col("n_valores") == 1,
           F.col("valor_total_adjudicacion").cast("decimal(20,2)")).alias("valor_adjudicado"),
    (F.col("n_valores") > 1).alias("adjudicacion_multiple"),
    F.col("respuestas_al_procedimiento").cast("int").alias("num_respuestas"),
    F.col("proveedores_unicos_con").cast("int").alias("num_proveedores_unicos"),
    F.col("duracion").cast("int").alias("duracion"),
    "unidad_de_duracion", "_fecha_ingesta")
  .dropDuplicates(["id_del_proceso"]))

regla = F.col("fecha_publicacion").isNotNull() & (F.col("precio_base") > 0)
validos, invalidos = procesos.filter(regla), procesos.filter(~regla)

invalidos.write.mode("overwrite").saveAsTable(f"{CATALOG}.secop_silver.procesos_cuarentena")
print(f"Válidos: {validos.count():,} | Cuarentena: {invalidos.count():,}")

## 3. Guardar los procesos sin crear duplicados

Usamos una operación llamada **MERGE**, que funciona así:
- Si el proceso ya existe en la tabla, lo actualiza.
- Si es nuevo, lo agrega.

Gracias a esto, el notebook se puede ejecutar varias veces sin que la tabla crezca con registros repetidos. Lo comprobamos ejecutando esta celda dos veces: el total no cambió.

In [0]:
# Merge IDEMPOTENTE
validos.createOrReplaceTempView("procesos_nuevos")
spark.sql(f"CREATE TABLE IF NOT EXISTS {CATALOG}.secop_silver.procesos "
          f"AS SELECT * FROM procesos_nuevos WHERE 1 = 0")
spark.sql(f"""
  MERGE INTO {CATALOG}.secop_silver.procesos t
  USING procesos_nuevos s ON t.id_del_proceso = s.id_del_proceso
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")
print(spark.table(f"{CATALOG}.secop_silver.procesos").count())

## 4. Tabla de adjudicaciones

Separamos en otra tabla la información de **quién ganó cada proceso**. Un proceso puede tener varios ganadores, por eso no cabe en la tabla de procesos, donde cada proceso ocupa una sola fila.

No incluimos el valor adjudicado aquí, porque la fuente no permite saber con certeza cuánto le correspondió a cada proveedor.

In [0]:
# Adjudicaciones
(b.filter(F.col("id_adjudicacion") != "No Adjudicado")
  .select("id_del_proceso", "id_adjudicacion", "codigoproveedor",
          "nit_del_proveedor_adjudicado",
          F.upper(F.trim("nombre_del_proveedor")).alias("proveedor"),
          "ciudad_proveedor", "departamento_proveedor")
  .dropDuplicates()
  .write.mode("overwrite").saveAsTable(f"{CATALOG}.secop_silver.adjudicaciones"))

## 5. Verificación

Comprobamos que no se perdió ningún proceso en el camino: los procesos válidos más los que están en cuarentena deben sumar exactamente el total de procesos únicos de Bronze.

| Tabla | Registros |
|---|---|
| Procesos válidos | 29.410 |
| En cuarentena | 508 |
| **Total** | **29.918** ✓ |
| Adjudicaciones | 1.123 |
| Procesos con adjudicación múltiple | 48 |

In [0]:
s = f"{CATALOG}.secop_silver"
n_proc = spark.table(f"{s}.procesos").count()
n_cuar = spark.table(f"{s}.procesos_cuarentena").count()
n_adj  = spark.table(f"{s}.adjudicaciones").count()
n_mult = spark.table(f"{s}.procesos").filter("adjudicacion_multiple").count()

print(f"Procesos: {n_proc:,} | Cuarentena: {n_cuar:,} | Suma: {n_proc + n_cuar:,} (esperado 29.918)")
print(f"Adjudicaciones: {n_adj:,} | Procesos con adjudicación múltiple: {n_mult:,}")